# 02. Dados Meteorológicos — ERA5 (ECMWF) + INMET

Extração das variáveis meteorológicas relevantes para propagação de incêndios via **CDS API** (ERA5) e **portal INMET**.

**Fonte primária:** ERA5 Reanalysis — https://cds.climate.copernicus.eu/  
**Fonte secundária:** INMET Dados Históricos — https://portal.inmet.gov.br/dadoshistoricos  
**Variáveis:** vento 10 m (u10, v10), temperatura 2 m (t2m), ponto de orvalho (d2m), precipitação (tp)  
**Período:** 2023 completo — mesmo período do notebook 01 (focos de calor INPE)

## 0. Dependências e configuração

> **Pré-requisito ERA5:** crie `~/.cdsapirc` com sua chave do Copernicus CDS:
> ```
> url: https://cds.climate.copernicus.eu/api
> key: SEU_UID:SUA_API_KEY
> ```
> Cadastro e token pessoal em https://cds.climate.copernicus.eu/profile  
> Instruções completas em https://cds.climate.copernicus.eu/how-to-api

In [ ]:
import calendar, zipfile
import requests
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

try:
    import cdsapi
    CDS_OK = True
except ImportError:
    CDS_OK = False
    print('[aviso] cdsapi nao instalado — execute: pip install cdsapi')

ERA5_DIR  = Path('data/era5')
INMET_DIR = Path('data/inmet')
ERA5_DIR.mkdir(parents=True, exist_ok=True)
INMET_DIR.mkdir(parents=True, exist_ok=True)

ANO = 2023

# Extensao do Brasil — mesma usada no notebook 01 (focos INPE)
# Formato ERA5: [N, W, S, E]
BBOX_ERA5 = [5.5, -74.0, -33.8, -28.6]

MESES_PT = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

print('Configuracao OK')
print(f'ERA5_DIR : {ERA5_DIR}')
print(f'INMET_DIR: {INMET_DIR}')
print(f'Periodo  : {ANO}')
print(f'BBOX     : {BBOX_ERA5}')

## 1. Download mensal via CDS API (ERA5)

In [ ]:
VARIAVEIS_ERA5 = [
    '10m_u_component_of_wind',   # u10  — componente zonal do vento
    '10m_v_component_of_wind',   # v10  — componente meridional do vento
    '2m_temperature',            # t2m  — temperatura do ar
    '2m_dewpoint_temperature',   # d2m  — ponto de orvalho (-> umidade relativa)
    'total_precipitation',       # tp   — precipitacao acumulada
]


def download_era5_mes(ano, mes, bbox=BBOX_ERA5, destino=ERA5_DIR):
    # Baixa ERA5 de um mes completo (4 horarios/dia: 00, 06, 12, 18 UTC) via CDS API
    # Solicitar todos os 24h/dia resulta em ~800 MB/mes — 4h/dia e suficiente para
    # medias diarias e reduz o volume ~6x (~130 MB/mes).
    #
    # Pre-requisito: ~/.cdsapirc com a nova API (migrada em set/2024):
    #   url: https://cds.climate.copernicus.eu/api
    #   key: <seu-personal-access-token>
    # (O formato antigo "uid:key" e a URL /api/v2 nao funcionam mais)
    ultimo_dia = calendar.monthrange(ano, mes)[1]
    arq = destino / f'era5_{ano}_{mes:02d}_brasil.nc'
    if arq.exists():
        print(f'  [cache] {arq.name} ({arq.stat().st_size/1024:.0f} KB)')
        return arq
    if not CDS_OK:
        raise RuntimeError('cdsapi nao disponivel')
    c = cdsapi.Client()
    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable'    : VARIAVEIS_ERA5,
            'year'        : str(ano),
            'month'       : f'{mes:02d}',
            'day'         : [f'{d:02d}' for d in range(1, ultimo_dia + 1)],
            'time'        : ['00:00', '06:00', '12:00', '18:00'],  # 4x/dia em vez de 24x
            'area'        : bbox,
            'format'      : 'netcdf',
        },
        str(arq),
    )
    print(f'  [ok] {arq.name} ({arq.stat().st_size/1024:.0f} KB)')
    return arq


print(f'Baixando ERA5 {ANO} (Brasil)...')
arquivos_era5 = []
for mes in range(1, 13):
    try:
        arquivos_era5.append(download_era5_mes(ANO, mes))
    except Exception as e:
        print(f'  [erro] {ANO}-{mes:02d}: {e}')

print(f'Arquivos baixados: {len(arquivos_era5)}/12')

## 2. Carregamento e variáveis derivadas

In [ ]:
# Carrega todos os arquivos mensais em um unico Dataset
ds = xr.open_mfdataset(
    sorted(arquivos_era5),
    combine='by_coords',
    chunks={'time': 4},   # 4 snapshots/dia (00, 06, 12, 18)
)

print(ds)
print(f'\nVariaveis: {list(ds.data_vars)}')
print(f'Periodo  : {str(ds.time.values[0])[:10]} -> {str(ds.time.values[-1])[:10]}')
print(f'Grade    : {ds.dims}')

In [ ]:
# Variaveis derivadas adicionadas ao ds (usadas nos mapas espaciais abaixo)

# Velocidade do vento (m/s)
ds['wind_speed'] = np.sqrt(ds['u10']**2 + ds['v10']**2)
ds['wind_speed'].attrs = {'units': 'm/s', 'long_name': 'Velocidade do vento 10 m'}

# Direcao do vento (graus, convencao meteorologica: de onde sopra)
ds['wind_dir'] = (np.degrees(np.arctan2(-ds['u10'], -ds['v10'])) % 360)
ds['wind_dir'].attrs = {'units': 'graus', 'long_name': 'Direcao do vento 10 m'}

# Temperatura em Celsius
ds['temp_c'] = ds['t2m'] - 273.15
ds['temp_c'].attrs = {'units': 'C', 'long_name': 'Temperatura 2 m'}

# Umidade relativa (formula de Magnus)
T  = ds['t2m']  - 273.15
Td = ds['d2m']  - 273.15
ds['rh'] = 100 * (np.exp((17.625 * Td) / (243.04 + Td)) /
                  np.exp((17.625 * T)  / (243.04 + T)))
ds['rh'].attrs = {'units': '%', 'long_name': 'Umidade relativa 2 m'}

# Indice de Vento Fosberg (FFWI)
eta = 1 - 2 * (ds['rh'] / 100) + (ds['rh'] / 100) ** 2
ds['ffwi'] = ds['wind_speed'] * np.sqrt(eta)
ds['ffwi'].attrs = {'units': 'adimensional', 'long_name': 'Fosberg Fire Weather Index'}

print('Variaveis derivadas calculadas:')
for v in ['wind_speed', 'wind_dir', 'temp_c', 'rh', 'ffwi']:
    print(f'  {v}: {ds[v].attrs.get("long_name", "")} [{ds[v].attrs.get("units", "")}]')

In [ ]:
# Media diaria sobre o dominio Brasil -> serie temporal
# Processa mes a mes para evitar carregar todos os dados na RAM de uma vez
dfs = []
for arq in sorted(arquivos_era5):
    ds_m = xr.open_dataset(arq)

    # Variaveis derivadas para este mes
    ws  = np.sqrt(ds_m['u10']**2 + ds_m['v10']**2)
    tc  = ds_m['t2m'] - 273.15
    td  = ds_m['d2m'] - 273.15
    rh  = 100 * (np.exp((17.625 * td) / (243.04 + td)) /
                 np.exp((17.625 * tc) / (243.04 + tc)))
    eta = 1 - 2 * (rh / 100) + (rh / 100) ** 2
    ff  = ws * np.sqrt(eta)
    pr  = ds_m['tp'] * 1000  # m -> mm

    # Media espacial (Brasil) e resample diario
    def _diario(da, how='mean'):
        sp = da.mean(dim=['latitude', 'longitude'])
        return sp.resample(time='1D').sum() if how == 'sum' else sp.resample(time='1D').mean()

    df_m = pd.DataFrame({
        'time'      : _diario(ws).time.values,
        'wind_speed': _diario(ws).values,
        'temp_c'    : _diario(tc).values,
        'rh'        : _diario(rh).values,
        'precip_mm' : _diario(pr, 'sum').values,
        'ffwi'      : _diario(ff).values,
    })
    dfs.append(df_m)
    ds_m.close()

df_serie = pd.concat(dfs, ignore_index=True)
df_serie['mes'] = pd.to_datetime(df_serie['time']).dt.month

print(f'Shape serie diaria: {df_serie.shape}')
print(f'Periodo: {pd.to_datetime(df_serie["time"]).min().date()} -> {pd.to_datetime(df_serie["time"]).max().date()}')
df_serie[['wind_speed','temp_c','rh','precip_mm','ffwi']].describe().round(2)

## 3. Visualizações de verificação

In [ ]:
# Painel 2x2 — medias mensais sobre o Brasil
CORES = {'wind_speed': '#1f77b4', 'temp_c': '#d62728', 'rh': '#2ca02c', 'precip_mm': '#9467bd', 'ffwi': '#ff7f0e'}

mens = df_serie.groupby('mes')[['wind_speed','temp_c','rh','precip_mm','ffwi']].mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'ERA5 — Variaveis Meteorologicas {ANO} (media Brasil)', fontsize=14, fontweight='bold')

# Vento
axes[0,0].bar(mens.index, mens['wind_speed'], color=CORES['wind_speed'], edgecolor='white')
axes[0,0].set_xticks(range(1,13)); axes[0,0].set_xticklabels(MESES_PT, fontsize=8)
axes[0,0].set_title('Velocidade Media do Vento (10 m)')
axes[0,0].set_ylabel('m/s')

# Temperatura
axes[0,1].bar(mens.index, mens['temp_c'], color=CORES['temp_c'], edgecolor='white')
axes[0,1].set_xticks(range(1,13)); axes[0,1].set_xticklabels(MESES_PT, fontsize=8)
axes[0,1].set_title('Temperatura Media (2 m)')
axes[0,1].set_ylabel('°C')

# Umidade relativa
axes[1,0].bar(mens.index, mens['rh'], color=CORES['rh'], edgecolor='white')
axes[1,0].set_xticks(range(1,13)); axes[1,0].set_xticklabels(MESES_PT, fontsize=8)
axes[1,0].set_title('Umidade Relativa Media (2 m)')
axes[1,0].set_ylabel('%')
axes[1,0].axhline(40, color='red', linestyle='--', linewidth=0.8, label='Limiar critico 40%')
axes[1,0].legend(fontsize=8)

# Precipitacao
axes[1,1].bar(mens.index, mens['precip_mm'], color=CORES['precip_mm'], edgecolor='white')
axes[1,1].set_xticks(range(1,13)); axes[1,1].set_xticklabels(MESES_PT, fontsize=8)
axes[1,1].set_title('Precipitacao Media Diaria')
axes[1,1].set_ylabel('mm/dia')

plt.tight_layout()
plt.savefig(ERA5_DIR / f'fig_painel_era5_{ANO}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Mapas espaciais — medias JJA e SON (estacoes de pico de incendio)
# JJA = Jun-Jul-Ago | SON = Set-Out-Nov  (periodo com mais focos no notebook 01)

def media_estacao(ds_var, meses):
    return ds_var.sel(time=ds_var.time.dt.month.isin(meses)).mean('time').compute()

wind_jja = media_estacao(ds['wind_speed'], [6, 7, 8])
wind_son = media_estacao(ds['wind_speed'], [9, 10, 11])
ffwi_jja = media_estacao(ds['ffwi'], [6, 7, 8])
ffwi_son = media_estacao(ds['ffwi'], [9, 10, 11])

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle(f'ERA5 {ANO} — Variaveis Espaciais (periodo de pico de incendios)', fontsize=13, fontweight='bold')

kw = dict(cmap='YlOrRd', add_colorbar=True)
wind_jja.plot(ax=axes[0,0], **kw); axes[0,0].set_title('Vento 10 m — JJA (m/s)')
wind_son.plot(ax=axes[0,1], **kw); axes[0,1].set_title('Vento 10 m — SON (m/s)')

kw_ffwi = dict(cmap='hot_r', add_colorbar=True)
ffwi_jja.plot(ax=axes[1,0], **kw_ffwi); axes[1,0].set_title('FFWI — JJA')
ffwi_son.plot(ax=axes[1,1], **kw_ffwi); axes[1,1].set_title('FFWI — SON')

for ax in axes.flat:
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

plt.tight_layout()
plt.savefig(ERA5_DIR / f'fig_mapas_era5_{ANO}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Series temporais diarias — cotejo com pico de focos (notebook 01)
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle(f'ERA5 — Series Temporais Diarias (media Brasil) — {ANO}', fontsize=13, fontweight='bold')

t = df_serie['time']

# Vento
axes[0].fill_between(t, df_serie['wind_speed'], alpha=0.35, color='#1f77b4')
axes[0].plot(t, df_serie['wind_speed'], color='#1f77b4', linewidth=0.8)
axes[0].set_ylabel('Vento (m/s)')
axes[0].set_title('Velocidade do Vento 10 m')
axes[0].grid(True, alpha=0.25)

# Temperatura e umidade relativa (eixo duplo)
ax_t = axes[1]
ax_rh = ax_t.twinx()
ax_t.plot(t, df_serie['temp_c'],  color='#d62728', linewidth=0.9, label='Temp. (°C)')
ax_rh.plot(t, df_serie['rh'],     color='#2ca02c', linewidth=0.9, label='UR (%)', alpha=0.8)
ax_t.set_ylabel('Temperatura (°C)', color='#d62728')
ax_rh.set_ylabel('Umidade relativa (%)', color='#2ca02c')
ax_t.axhline(df_serie['temp_c'].mean(), color='#d62728', linestyle='--', linewidth=0.7)
ax_rh.axhline(40, color='#2ca02c', linestyle='--', linewidth=0.7, label='Limiar UR 40%')
lines1, lab1 = ax_t.get_legend_handles_labels()
lines2, lab2 = ax_rh.get_legend_handles_labels()
ax_t.legend(lines1+lines2, lab1+lab2, fontsize=8, loc='upper left')
ax_t.set_title('Temperatura e Umidade Relativa')
ax_t.grid(True, alpha=0.25)

# FFWI
axes[2].fill_between(t, df_serie['ffwi'], alpha=0.45, color='#ff7f0e')
axes[2].plot(t, df_serie['ffwi'], color='#ff7f0e', linewidth=0.8)
pico_ffwi = df_serie.loc[df_serie['ffwi'].idxmax()]
axes[2].annotate(
    f"Pico FFWI: {pico_ffwi['ffwi']:.1f}\n{pico_ffwi['time'].strftime('%d/%m/%Y')}",
    xy=(pico_ffwi['time'], pico_ffwi['ffwi']),
    xytext=(30, -15), textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color='black'), fontsize=8
)
axes[2].set_ylabel('FFWI')
axes[2].set_title('Fosberg Fire Weather Index (proxy de perigo de incendio)')
axes[2].grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig(ERA5_DIR / f'fig_serie_era5_{ANO}.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. INMET — Validação Pontual

In [ ]:
# Download do arquivo anual do INMET (dados historicos — estacoes automaticas)
# Fonte: https://portal.inmet.gov.br/dadoshistoricos

def download_inmet_ano(ano, destino=INMET_DIR):
    url      = f'https://portal.inmet.gov.br/uploads/dadoshistoricos/{ano}.zip'
    zip_path = destino / f'{ano}.zip'
    if not zip_path.exists():
        print(f'Baixando INMET {ano}...', end=' ', flush=True)
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        zip_path.write_bytes(resp.content)
        print(f'{zip_path.stat().st_size/1e6:.1f} MB')
    extract_dir = destino / str(ano)
    if not extract_dir.exists():
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)
        print(f'  [ok] Extraido em {extract_dir}')
    else:
        print(f'  [cache] {extract_dir}')
    csvs = sorted(extract_dir.rglob('*.CSV')) + sorted(extract_dir.rglob('*.csv'))
    print(f'  Estacoes encontradas: {len(csvs)}')
    return csvs


csvs_inmet = download_inmet_ano(ANO)

In [ ]:
def ler_estacao_inmet(caminho):
    meta = {}
    with open(caminho, encoding='latin-1') as f:
        for _ in range(8):
            linha = f.readline().strip()
            if ':' in linha:
                k, v = linha.split(':', 1)
                meta[k.strip()] = v.strip()
    df = pd.read_csv(
        caminho, skiprows=8, sep=';', encoding='latin-1',
        decimal=',', na_values=['-9999', ''], low_memory=False
    )
    df['estacao'] = meta.get('Codigo Estacao', '')
    df['lat']     = float(str(meta.get('Latitude',  '0')).replace(',', '.'))
    df['lon']     = float(str(meta.get('Longitude', '0')).replace(',', '.'))
    return df, meta


# Carrega todas as estacoes — metadados (lat/lon/uf)
registros_meta = []
for csv in csvs_inmet:
    try:
        _, meta = ler_estacao_inmet(csv)
        registros_meta.append({
            'arquivo' : csv,
            'codigo'  : meta.get('Codigo Estacao', ''),
            'nome'    : meta.get('Nome', ''),
            'uf'      : meta.get('UF', ''),
            'lat'     : float(str(meta.get('Latitude', '0')).replace(',', '.')),
            'lon'     : float(str(meta.get('Longitude', '0')).replace(',', '.')),
        })
    except Exception:
        pass

df_meta = pd.DataFrame(registros_meta)
print(f'Estacoes com metadados: {len(df_meta)}')
print(f'UFs cobertas: {sorted(df_meta["uf"].unique())}')
df_meta.head()

In [ ]:
# Seleciona estacao proxima ao centroide do Cerrado (maior densidade de focos — notebook 01)
# Referencia: Brasilia/DF ~ lat=-15.8, lon=-47.9
LAT_REF, LON_REF = -15.8, -47.9
df_meta['dist'] = np.sqrt((df_meta['lat'] - LAT_REF)**2 + (df_meta['lon'] - LON_REF)**2)
est_ref = df_meta.loc[df_meta['dist'].idxmin()]
print(f'Estacao de referencia: {est_ref["codigo"]} — {est_ref["nome"]} ({est_ref["uf"]})')
print(f'Lat: {est_ref["lat"]:.4f} | Lon: {est_ref["lon"]:.4f} | Dist: {est_ref["dist"]:.2f} graus')

df_est, _ = ler_estacao_inmet(est_ref['arquivo'])

# Identifica coluna de data
col_data = [c for c in df_est.columns if 'DATA' in c.upper() or 'DATA' in c][0]
col_hora = [c for c in df_est.columns if 'HORA' in c.upper() or 'UTC' in c.upper()]
col_hora = col_hora[0] if col_hora else None

df_est['datetime'] = pd.to_datetime(
    df_est[col_data].astype(str) + ' ' + df_est[col_hora].astype(str) if col_hora else df_est[col_data].astype(str),
    errors='coerce'
)
df_est = df_est.dropna(subset=['datetime']).sort_values('datetime')
print(f'\nRegistros: {len(df_est):,}')
print(f'Periodo  : {df_est["datetime"].min()} -> {df_est["datetime"].max()}')
print(f'Colunas  : {df_est.columns.tolist()}')

In [ ]:
# Identifica colunas de temperatura, umidade e vento (variam por versao INMET)
def achar_col(df, termos):
    for t in termos:
        cols = [c for c in df.columns if t.upper() in c.upper()]
        if cols:
            return cols[0]
    return None

col_temp  = achar_col(df_est, ['TEMPERATURA DO AR', 'TEMP. AR', 'TEMP AR'])
col_ur    = achar_col(df_est, ['UMIDADE RELATIVA', 'UMIDADE REL', 'UMD. REL.'])
col_vento = achar_col(df_est, ['VENTO, VELOCIDADE', 'VELOC. VENTO', 'VEL. VENTO'])
col_prec  = achar_col(df_est, ['PRECIPITACAO', 'PRECIPIT', 'CHUVA'])

print('Colunas identificadas:')
print(f'  Temperatura : {col_temp}')
print(f'  Umidade rel.: {col_ur}')
print(f'  Vento       : {col_vento}')
print(f'  Precipitacao: {col_prec}')

# Media diaria
df_est['data'] = df_est['datetime'].dt.date
agg_cols = {c: 'mean' for c in [col_temp, col_ur, col_vento] if c}
if col_prec:
    agg_cols[col_prec] = 'sum'

df_diario_inmet = (
    df_est.assign(**{c: pd.to_numeric(df_est[c], errors='coerce') for c in agg_cols})
           .groupby('data')
           .agg(agg_cols)
           .reset_index()
)
df_diario_inmet['data'] = pd.to_datetime(df_diario_inmet['data'])
print(f'\nSerie diaria INMET: {len(df_diario_inmet)} dias')
df_diario_inmet.head()

In [ ]:
# Cotejo ERA5 x INMET — ponto mais proximo na grade ERA5
era5_ponto = ds[['temp_c', 'rh', 'wind_speed']].sel(
    latitude=est_ref['lat'],
    longitude=est_ref['lon'],
    method='nearest'
).resample(time='1D').mean().compute().to_dataframe().reset_index()

if col_temp and col_ur:
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    fig.suptitle(f'Cotejo ERA5 x INMET — Estacao {est_ref["codigo"]} ({est_ref["nome"]}) — {ANO}',
                 fontsize=12, fontweight='bold')

    axes[0].plot(era5_ponto['time'], era5_ponto['temp_c'],
                 label='ERA5', color='#1f77b4', linewidth=0.9)
    axes[0].plot(df_diario_inmet['data'], pd.to_numeric(df_diario_inmet[col_temp], errors='coerce'),
                 label='INMET', color='#d62728', linewidth=0.9, alpha=0.8)
    axes[0].set_ylabel('Temperatura (°C)')
    axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.25)

    axes[1].plot(era5_ponto['time'], era5_ponto['rh'],
                 label='ERA5', color='#1f77b4', linewidth=0.9)
    axes[1].plot(df_diario_inmet['data'], pd.to_numeric(df_diario_inmet[col_ur], errors='coerce'),
                 label='INMET', color='#d62728', linewidth=0.9, alpha=0.8)
    axes[1].set_ylabel('Umidade Relativa (%)')
    axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.25)

    plt.tight_layout()
    plt.savefig(ERA5_DIR / f'fig_cotejo_era5_inmet_{ANO}.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('[aviso] Colunas de temperatura/umidade nao identificadas no INMET — verifique o formato do CSV.')

## 5. Estatísticas resumo e exportação

In [ ]:
print(f'Estatisticas gerais ERA5 — {ANO} (media diaria Brasil)')
estatisticas = df_serie[['wind_speed','temp_c','rh','precip_mm','ffwi']].describe().round(2)
print(estatisticas.to_string())

print()
print('Media mensal')
mens_fmt = mens.copy()
mens_fmt.index = MESES_PT
mens_fmt.columns = ['Vento (m/s)', 'Temp (°C)', 'UR (%)', 'Precip (mm/d)', 'FFWI']
print(mens_fmt.round(2).to_string())

print()
print(f'=== Periodo de pico de incendios (Set-Out {ANO} — notebook 01) ===')
peak = df_serie[df_serie['mes'].isin([9, 10])]
print(f'  Vento medio   : {peak["wind_speed"].mean():.2f} m/s')
print(f'  Temp. media   : {peak["temp_c"].mean():.1f} °C')
print(f'  UR media      : {peak["rh"].mean():.1f} %')
print(f'  Precip. media : {peak["precip_mm"].mean():.2f} mm/dia')
print(f'  FFWI medio    : {peak["ffwi"].mean():.2f}')

In [ ]:
# Exporta serie diaria consolidada (usada nos proximos notebooks)
out_csv = ERA5_DIR / f'era5_{ANO}_brasil_diario.csv'
df_serie.to_csv(out_csv, index=False)
print(f'Dataset salvo: {out_csv} ({out_csv.stat().st_size/1e3:.1f} KB)')

# Exporta metadados das estacoes INMET
out_meta = INMET_DIR / f'inmet_{ANO}_estacoes_meta.csv'
df_meta.drop(columns=['arquivo','dist'], errors='ignore').to_csv(out_meta, index=False)
print(f'Metadados INMET salvos: {out_meta}')

print()
print('Arquivos gerados')
for p in sorted(ERA5_DIR.iterdir()):
    print(f'  {p.name:50s} {p.stat().st_size/1024:8.0f} KB')